<a href="https://colab.research.google.com/github/contreras-juan/procesamiento_lenguaje_natural/blob/main/Ejercicios/04_Taller_HMM_MEMM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="color: #FECB05; text-align: center;">Taller práctico: HMM, Viterbi y MEMM</h1>

<h2 style="color: #007ACC;">Autores</h2>

- [Juan Felipe Contreras Alcívar](https://www.linkedin.com/in/juanf-contreras/)

---

<h2 style="color: #007ACC;">Instrucciones</h2>

Este taller practica los temas de `08_HMM.ipynb`, `09_Viterbi.ipynb` y `10_MEMM.ipynb` con **otra tarea y otros datos**: en lugar de etiquetado gramatical, haremos **reconocimiento de entidades nombradas** (NER) en español con el corpus **CoNLL-2002**, noticias de la agencia EFE anotadas con personas (`PER`), lugares (`LOC`), organizaciones (`ORG`) y otras entidades (`MISC`) en formato BIO.

Enfocamos:

- la evaluación **por entidades** (no por tokens) con precisión, *recall* y F1,
- el algoritmo forward en **espacio logarítmico**,
- un HMM con clases de **forma de palabra** (*word shape*), muy útiles para NER,
- la implementación de **Viterbi** y su uso con un HMM y con un MEMM,
- el análisis de errores propio de NER: tipo equivocado frente a límites equivocados.

CoNLL-2002 trae tres particiones: `esp.train` (entrenamiento), `esp.testa` (validación) y `esp.testb` (prueba). Úsalas en ese orden: el conjunto de prueba solo se usa al final.

<h2 style="color: #007ACC;">Tabla de contenido</h2>

- [<span style="color: #005C99;">Ejercicio 1. Entidades a partir de etiquetas BIO y evaluación</span>](#ejercicio-1)
- [<span style="color: #005C99;">Ejercicio 2. El algoritmo forward en espacio logarítmico</span>](#ejercicio-2)
- [<span style="color: #005C99;">Ejercicio 3. Un HMM para NER con formas de palabra</span>](#ejercicio-3)
- [<span style="color: #005C99;">Ejercicio 4. Viterbi para NER</span>](#ejercicio-4)
- [<span style="color: #005C99;">Ejercicio 5. Un MEMM para NER</span>](#ejercicio-5)
- [<span style="color: #005C99;">Ejercicio 6. Análisis de errores</span>](#ejercicio-6)

---

<h2 style="color: #007ACC;">Instalación de librerías</h2>

Ejecuta esta celda si trabajas en Google Colab o en un entorno nuevo.

In [ ]:
# Instalación de librerías necesarias para esta sesión
# (útil en Google Colab o en un entorno nuevo)
%pip install -q numpy pandas matplotlib scipy scikit-learn nltk

In [ ]:
import itertools
import time
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import logsumexp

import nltk
from nltk.corpus import conll2002

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

SEED = 42
np.random.seed(SEED)

_ = nltk.download("conll2002", quiet=True)


def cargar(particion):
    """Lista de oraciones; cada oración es una lista de (palabra, etiqueta BIO)."""
    return [[(w, ner) for w, _, ner in s] for s in conll2002.iob_sents(particion)]


train, dev, test = cargar("esp.train"), cargar("esp.testa"), cargar("esp.testb")
ETIQUETAS = sorted({t for s in train for _, t in s})
t2i = {t: i for i, t in enumerate(ETIQUETAS)}
print(f"Oraciones: train={len(train)}, dev={len(dev)}, test={len(test)}")
print("Etiquetas:", ETIQUETAS)
print(train[2][:15])

---

<a id="ejercicio-1"></a>
<h2 style="color: #007ACC;">Ejercicio 1. Entidades a partir de etiquetas BIO y evaluación</h2>

**Objetivo:** convertir secuencias BIO en entidades y evaluar con precisión, *recall* y F1 por entidad, como en la tarea compartida CoNLL.

1. Calcula la distribución de etiquetas en entrenamiento. ¿Qué exactitud por token obtendría un etiquetador que siempre predice `O`?
2. Implementa `entidades(tags)` que devuelva el **conjunto** de entidades de una oración como tuplas `(inicio, fin, tipo)` (con `fin` exclusivo). Sigue la convención del script oficial `conlleval`: una entidad empieza con `B-X`, o con `I-X` si la etiqueta anterior no era del mismo tipo `X`.
3. Implementa `evaluar_entidades(oraciones_reales, oraciones_predichas)` que devuelva precisión, *recall* y F1 **micro** (sumando sobre todo el corpus). Una entidad predicha cuenta como correcta solo si coinciden el inicio, el fin **y** el tipo. Devuelve también el F1 por tipo de entidad.
4. Verifica con este ejemplo: reales `B-PER I-PER O B-LOC`, predichas `B-PER O O B-ORG` (precisión 0, recall 0: los límites de la persona y el tipo del lugar están mal).
5. ¿Por qué la exactitud por token es una mala métrica para NER?

In [ ]:
# Escribe tu código aquí
def entidades(tags):
    pass


def evaluar_entidades(reales, predichas):
    """reales y predichas: listas de secuencias de etiquetas (una por oración)."""
    pass

---

<a id="ejercicio-2"></a>
<h2 style="color: #007ACC;">Ejercicio 2. El algoritmo forward en espacio logarítmico</h2>

**Objetivo:** implementar forward con log-probabilidades y comprobar por qué es necesario.

Usa este HMM de juguete con 3 estados y 4 símbolos:

```python
pi_j = np.array([0.6, 0.3, 0.1])
A_j = np.array([[0.7, 0.2, 0.1], [0.3, 0.5, 0.2], [0.2, 0.3, 0.5]])
B_j = np.array([[0.5, 0.3, 0.1, 0.1], [0.1, 0.4, 0.4, 0.1], [0.1, 0.1, 0.2, 0.6]])
```

1. Implementa `forward(obs, pi, A, B)` en probabilidades (como en el cuaderno 08) y `forward_log(obs, log_pi, log_A, log_B)` usando `scipy.special.logsumexp`. Pista: la recursión es $\log \alpha_t(j) = \text{logsumexp}_i\left(\log \alpha_{t-1}(i) + \log a_{ij}\right) + \log b_j(o_t)$.
2. Verifica ambas contra la fuerza bruta para una secuencia de longitud 6.
3. Genera una secuencia aleatoria de longitud 1 000 y calcula $P(O)$ con ambas versiones. ¿Qué ocurre y por qué?

In [ ]:
pi_j = np.array([0.6, 0.3, 0.1])
A_j = np.array([[0.7, 0.2, 0.1], [0.3, 0.5, 0.2], [0.2, 0.3, 0.5]])
B_j = np.array([[0.5, 0.3, 0.1, 0.1], [0.1, 0.4, 0.4, 0.1], [0.1, 0.1, 0.2, 0.6]])

# Escribe tu código aquí
def forward(obs, pi, A, B):
    pass


def forward_log(obs, log_pi, log_A, log_B):
    pass

---

<a id="ejercicio-3"></a>
<h2 style="color: #007ACC;">Ejercicio 3. Un HMM para NER con formas de palabra</h2>

**Objetivo:** estimar un HMM supervisado para NER en el que las palabras raras y desconocidas se representan por su forma.

En NER, la **forma** de una palabra es una pista muy fuerte: *Madrid* (`Xx`), *ONU* (`X`), *1998* (`d`), *Juan-Carlos* (`Xx-Xx`)...

1. Implementa `forma(palabra)` que reemplace las mayúsculas por `X`, las minúsculas por `x` y los dígitos por `d`, colapsando las repeticiones consecutivas: `"Madrid" → "Xx"`, `"ONU" → "X"`, `"1998" → "d"`, `"EE.UU." → "X.X."`, `"Juan-Carlos" → "Xx-Xx"`. Añade el prefijo `"<FORMA>"` para que no se confunda con una palabra real.
2. Define el vocabulario como las palabras (respetando mayúsculas) que aparecen al menos 2 veces en entrenamiento. Las demás, en entrenamiento y en evaluación, se reemplazan por su forma.
3. Estima $\pi$, $A$ y $B$ por conteo con un suavizado aditivo pequeño (por ejemplo $10^{-3}$) y guarda sus logaritmos.
4. Muestra las 5 transiciones más probables desde `B-PER`, `B-ORG` y `O`. ¿Aprendió el modelo la gramática del formato BIO (por ejemplo, que `I-PER` solo sigue a `B-PER` o `I-PER`)?
5. Muestra las formas más frecuentes que emite cada tipo de entidad.

In [ ]:
# Escribe tu código aquí
def forma(palabra):
    pass

---

<a id="ejercicio-4"></a>
<h2 style="color: #007ACC;">Ejercicio 4. Viterbi para NER</h2>

**Objetivo:** implementar Viterbi en espacio logarítmico, decodificar con el HMM y compararlo con una línea base.

1. Implementa `viterbi(log_pi, log_A, log_B)` donde `log_B` es la matriz $(T \times N)$ de log-emisiones de la oración. Devuelve la lista de índices de estados.
2. Implementa `etiquetar_hmm(palabras)` y etiqueta el conjunto de **validación**.
3. Implementa una **línea base**: cada palabra recibe la etiqueta con la que más apareció en entrenamiento; las desconocidas reciben `O`.
4. Compara ambos con la exactitud por token y con `evaluar_entidades` (F1 global y por tipo).
5. Cuenta cuántas transiciones BIO inválidas (`I-X` después de algo que no sea `B-X` o `I-X`) produce cada método. ¿Por qué el HMM casi no las produce?

In [ ]:
# Escribe tu código aquí
def viterbi(log_pi, log_A, log_B):
    pass

---

<a id="ejercicio-5"></a>
<h2 style="color: #007ACC;">Ejercicio 5. Un MEMM para NER</h2>

**Objetivo:** entrenar un MEMM con características ricas, decodificarlo con Viterbi y compararlo con el HMM.

1. Diseña `caracteristicas(palabras, i, etiqueta_previa)` para NER. Incluye al menos: la palabra en minúscula, su forma, prefijos y sufijos, si es la primera palabra de la oración, las palabras vecinas (y si empiezan con mayúscula) y la etiqueta anterior.
2. Entrena un `LogisticRegression` (con *teacher forcing*: la etiqueta anterior verdadera) sobre el conjunto de entrenamiento.
3. Implementa la decodificación con Viterbi del MEMM (como en el cuaderno 10) y etiqueta el conjunto de **validación**.
4. Compara con la línea base y el HMM usando la función `comparar` del ejercicio 4.
5. Elige el mejor modelo según **validación** y evalúalo **una sola vez** en el conjunto de **prueba** (`esp.testb`).

In [ ]:
# Escribe tu código aquí
def caracteristicas(palabras, i, etiqueta_previa):
    pass

---

<a id="ejercicio-6"></a>
<h2 style="color: #007ACC;">Ejercicio 6. Análisis de errores</h2>

**Objetivo:** clasificar los errores de NER por tipo y entender qué le falta al modelo.

Con las predicciones del MEMM sobre el conjunto de **validación**:

1. Clasifica cada entidad real no encontrada exactamente en una de estas categorías:
   - **tipo equivocado:** hay una entidad predicha con los mismos límites pero otro tipo;
   - **límites equivocados:** hay una entidad predicha que se solapa con la real pero con otros límites;
   - **omitida:** ninguna entidad predicha se solapa con la real.
2. Cuenta también los **falsos positivos** puros: entidades predichas que no se solapan con ninguna real.
3. Para los errores de tipo, construye una tabla real × predicho. ¿Qué confusiones dominan?
4. Muestra 8 ejemplos de errores con su contexto y comenta al menos tres.

In [ ]:
# Escribe tu código aquí

<a id="referencias"></a>
<h2 style="color: #007ACC;">Referencias</h2>

- Jurafsky, D. & Martin, J. H. *Speech and Language Processing* (3.ª ed. borrador). [web.stanford.edu/~jurafsky/slp3](https://web.stanford.edu/~jurafsky/slp3/)
- Tjong Kim Sang, E. F. (2002). Introduction to the CoNLL-2002 Shared Task: Language-Independent Named Entity Recognition. *CoNLL*.
- Carreras, X., Màrquez, L. & Padró, L. (2002). Named Entity Extraction using AdaBoost. *CoNLL*.
- McCallum, A., Freitag, D. & Pereira, F. (2000). Maximum Entropy Markov Models for Information Extraction and Segmentation. *ICML*.